In [5]:
import os
import pydicom
import numpy as np
import cv2

In [6]:
input_directory = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")
output_directory = os.path.join("..", "data", "PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED")
os.makedirs(output_directory, exist_ok=True)

In [7]:
color_threshold = 200
processed_count = 0

print("Starting batch segmentation with contour subtraction...")
print(f"Reading from: {os.path.abspath(input_directory)}")
print(f"Saving to:    {os.path.abspath(output_directory)}\n")

for study_id in sorted(os.listdir(input_directory)):
    if study_id.startswith("."):
        continue

    study_path = os.path.join(input_directory, study_id)

    date_folders = [f for f in os.listdir(study_path) if not f.startswith(".")]
    if not date_folders:
        continue
    date_path = os.path.join(study_path, date_folders[0])

    files = [f for f in os.listdir(date_path) if not f.startswith(".")]
    if not files:
        continue
    dicom_path = os.path.join(date_path, files[0])

    try:
        # Load the image
        dataset = pydicom.dcmread(dicom_path)
        image_rgb = dataset.pixel_array

        # Isolate white pixels
        white_mask = (image_rgb[:, :, 0] > color_threshold) & \
                     (image_rgb[:, :, 1] > color_threshold) & \
                     (image_rgb[:, :, 2] > color_threshold)
        white_mask_8bit = (white_mask * 255).astype(np.uint8)

        # Keep largest island only
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(white_mask_8bit, connectivity=8)

        largest_label = 1
        max_area = 0
        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if area > max_area:
                max_area = area
                largest_label = i

        clean_contour = np.zeros_like(white_mask_8bit)
        if max_area > 0:
            clean_contour[labels == largest_label] = 255

        # Close gaps in the contour
        kernel = np.ones((10, 10), np.uint8)
        closed_contour = cv2.morphologyEx(clean_contour, cv2.MORPH_CLOSE, kernel)

        # Fill the contour
        contours, hierarchy = cv2.findContours(closed_contour, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        filled_mask = np.zeros_like(closed_contour)
        cv2.drawContours(filled_mask, contours, -1, color=255, thickness=cv2.FILLED)

        # Subtract the contour outline from the filled mask
        clean_mask = filled_mask.copy()
        clean_mask[closed_contour > 0] = 0

        # Convert to grayscale
        gray_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

        # Save image and mask
        image_save_path = os.path.join(output_directory, f"{study_id}_image.png")
        mask_save_path = os.path.join(output_directory, f"{study_id}_mask.png")

        cv2.imwrite(image_save_path, gray_image)
        cv2.imwrite(mask_save_path, clean_mask)

        processed_count += 1

        if processed_count % 20 == 0:
            print(f"Processed {processed_count} images...")

    except Exception as e:
        print(f"Error processing {study_id}: {e}")

print(f"\nTotal processed: {processed_count}")
print(f"Output folder: {os.path.abspath(output_directory)}")

Starting batch segmentation with contour subtraction...
Reading from: /home/daniduhnev/projects/master-thesis/data/PANCREAS_2/PANCREAS_2
Saving to:    /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED

Processed 20 images...
Processed 40 images...
Processed 60 images...
Processed 80 images...
Processed 100 images...
Processed 120 images...

Total processed: 134
Output folder: /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED
